# Steam Hybrid AI Recommender - MERGED DEPLOYMENT V4 (Lazy Loading)

Gộp từ RS5_v5 + RecSas, tối ưu RAM cho Colab T4 (15GB).

## RAM Strategy — Lazy Loading:
- **Startup (~4-5GB):** Only Two-Tower, FAISS, NLP, LightGBM, game dicts
- **Lazy (~4-6GB on demand):** ALS, BM25, SASRec, user_feat_df
- **Always deleted:** game_docs_df → dicts, text_emb_matrix → SASRec weights
- **DuckDB:** read_parquet() directly (no CREATE TABLE)
- **RAM guard:** Warning if usage > 10GB
- **GAME_TOKEN_SETS:** Lazy-built on first reranker request

## Luồng chạy:
1. Install + Paths
2. Load Essential Models + Lazy System
3. Live Data Collection (Steam API, trending)
4. Core Engine: `get_perfect_hybrid_recommendation`
5. FastAPI Server + LocalTunnel

In [ ]:
# CELL 1: INSTALL + IMPORT + PATH CONFIG

!pip install -q duckdb tensorflow-recommenders faiss-gpu-cu12 lightgbm sentence-transformers rank-bm25 implicit beautifulsoup4 psutil
!pip install -q fastapi uvicorn nest-asyncio pydantic
!npm install -g localtunnel 2>/dev/null

import os
import re
import gc
import json
import time
import pickle
import threading
import traceback
import warnings
import psutil
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import duckdb
import requests
import scipy.sparse as sp
import lightgbm as lgb
import faiss

from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup

os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import tensorflow as tf

import torch
import torch.nn as nn

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ============================================================
# PATH CONFIG
# ============================================================
RS5_BASE_DIR = '/content/drive/MyDrive/steam/steam_100M/processed_data'
RS5_MODEL_DIR = f'{RS5_BASE_DIR}/models'
GAME_API_PATH = f'{RS5_BASE_DIR}/game_features_api_enriched.parquet'
GAME_FEAT_PATH = f'{RS5_BASE_DIR}/game_features_enhanced.parquet'
USER_FEAT_PATH = f'{RS5_BASE_DIR}/user_features_full.parquet'
FAISS_GAME_INDEX_PATH = f'{RS5_MODEL_DIR}/faiss_game_index.bin'
FAISS_GAME_APPIDS_PATH = f'{RS5_MODEL_DIR}/faiss_appids.npy'
TFRS_MODEL_PATH = f'{RS5_MODEL_DIR}/two_tower_user_model'
LGBM_MODEL_PATH = f'{RS5_MODEL_DIR}/lightgbm_ranker.txt'

RS_BASE_PATH = '/content/drive/MyDrive/steam/steam_base2'
RS_MODEL_DIR = f'{RS_BASE_PATH}/models_nlp_v2'
RS_CACHE_DIR = f'{RS_BASE_PATH}/cache_nlp_v2'
ADV_MODEL_DIR = f'{RS_BASE_PATH}/models_recsys_v3'
ADV_CACHE_DIR = f'{RS_BASE_PATH}/cache_recsys_v3'
os.makedirs(RS_MODEL_DIR, exist_ok=True)
os.makedirs(RS_CACHE_DIR, exist_ok=True)
os.makedirs(ADV_MODEL_DIR, exist_ok=True)
os.makedirs(ADV_CACHE_DIR, exist_ok=True)

TRAIN_PATH = f'{RS_BASE_PATH}/train_interactions.parquet'
VAL_PATH = f'{RS_BASE_PATH}/val_interactions.parquet'
GAME_DOCS_PATH = f'{RS_CACHE_DIR}/game_documents.parquet'
POPULAR_PATH = f'{RS_CACHE_DIR}/popular_games.parquet'
FAISS_NLP_INDEX_PATH = f'{RS_MODEL_DIR}/faiss_nlp_game_index.bin'
FAISS_NLP_APPIDS_PATH = f'{RS_MODEL_DIR}/faiss_nlp_appids.npy'
GAME_TEXT_EMB_PATH = f'{RS_MODEL_DIR}/game_text_embeddings.npy'
BM25_PATH = f'{RS_MODEL_DIR}/bm25_game_docs.pkl'
ALS_MODEL_PATH = f'{RS_MODEL_DIR}/als_model.pkl'
USER_ITEM_MATRIX_PATH = f'{RS_CACHE_DIR}/user_item_train.npz'
USER2IDX_PATH = f'{RS_CACHE_DIR}/user2idx.pkl'
ITEM2IDX_PATH = f'{RS_CACHE_DIR}/item2idx.pkl'
IDX2ITEM_PATH = f'{RS_CACHE_DIR}/idx2item.pkl'
ITEM2ITEM_PATH = f'{RS_CACHE_DIR}/item2item_topk.parquet'
TEXT_SASREC_MODEL_PATH = f'{ADV_MODEL_DIR}/text_enhanced_sasrec.pt'
RERANKER_V3_MODEL_PATH = f'{ADV_MODEL_DIR}/lightgbm_reranker_v3_sasrec.txt'
STEAM_API_KEY = '84FF2705598E2FD2B7B92901E691BA4A'

print(f'[OK] Paths configured. Device: {DEVICE}')

In [ ]:
# CELL 2: LOAD ESSENTIAL MODELS + LAZY LOADING SYSTEM
# RAM strategy: Only load Two-Tower, FAISS, NLP, LightGBM at startup.
# ALS, BM25, SASRec loaded on-demand. NSFW lazy-loaded on first request.
# Expected startup RAM: ~4-5GB (vs ~8-12GB before)

load_start = time.time()

def get_ram_gb():
    return psutil.virtual_memory().used / (1024**3)

def ram_guard(label, warn_gb=10):
    used = get_ram_gb()
    if used > warn_gb:
        print(f'  [RAM WARNING] {label}: {used:.1f}GB used!')
    return used

# ================================================================
# ESSENTIAL MODELS (loaded at startup — always needed)
# ================================================================

# --- 1. DuckDB (no table caching) ---
print('[1/6] DuckDB connection...')
con = duckdb.connect()
con.execute('PRAGMA threads=4')

# --- 2. Two-Tower Model ---
print('[2/6] Two-Tower model...')
user_model = tf.saved_model.load(TFRS_MODEL_PATH)
faiss_game_index = faiss.read_index(FAISS_GAME_INDEX_PATH)
faiss_game_appids = np.load(FAISS_GAME_APPIDS_PATH, allow_pickle=True)
ram_guard('Two-Tower')

# --- 3. LightGBM Ranker ---
print('[3/6] LightGBM ranker...')
ranker = lgb.Booster(model_file=LGBM_MODEL_PATH)

# --- 4. NLP Model ---
print('[4/6] SentenceTransformer...')
BASE_EMBEDDING_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
FINETUNED_ENCODER_DIR = f'{RS_MODEL_DIR}/steam_multilingual_text_encoder'
if os.path.exists(FINETUNED_ENCODER_DIR):
    nlp_model = SentenceTransformer(FINETUNED_ENCODER_DIR, device=DEVICE)
else:
    nlp_model = SentenceTransformer(BASE_EMBEDDING_MODEL, device=DEVICE)
ram_guard('NLP Model')

# --- 5. FAISS Text Index ---
print('[5/6] FAISS text index...')
faiss_text_index = None
faiss_text_appids = None
if os.path.exists(FAISS_NLP_INDEX_PATH):
    faiss_text_index = faiss.read_index(FAISS_NLP_INDEX_PATH)
    faiss_text_appids = np.load(FAISS_NLP_APPIDS_PATH, allow_pickle=True)
print(f'  FAISS text: {faiss_text_index.ntotal if faiss_text_index else 0:,}')
ram_guard('FAISS Text')

# --- 6. Game Documents + Lookup Dicts ---
print('[6/6] Game documents + lookup dicts...')
game_docs_df = pd.read_parquet(GAME_DOCS_PATH)
game_docs_df['appid'] = game_docs_df['appid'].astype(str)

appid_to_name = dict(zip(game_docs_df['appid'], game_docs_df['name'].fillna('')))
appid_to_tag_text = dict(zip(game_docs_df['appid'], (game_docs_df.get('genres', pd.Series(dtype=str)).fillna('') + ' ' + game_docs_df.get('categories', pd.Series(dtype=str)).fillna('')).str.lower()))
# appid_to_short DELETED — not used in recommendation logic

GAME_TOKEN_SETS = None
GAME_TOKEN_SETS_BUILT = False
HISTORY_TOKEN_CACHE = {}

del game_docs_df
gc.collect()
ram_guard('Game Dicts')

# ================================================================
# LAZY LOADING SYSTEM — load on demand, keep after first load
# ================================================================
_LAZY = {}

def _load_als():
    if 'als' in _LAZY: return _LAZY['als']
    print('  [LAZY] Loading ALS model + mappings...')
    with open(ALS_MODEL_PATH, 'rb') as f: als_model = pickle.load(f)
    with open(USER2IDX_PATH, 'rb') as f: user2idx = pickle.load(f)
    with open(ITEM2IDX_PATH, 'rb') as f: item2idx = pickle.load(f)
    with open(IDX2ITEM_PATH, 'rb') as f: idx2item = pickle.load(f)
    if isinstance(idx2item, np.ndarray): idx2item = idx2item.tolist()
    train_user_items = sp.load_npz(USER_ITEM_MATRIX_PATH).tocsr()
    _LAZY['als'] = (als_model, user2idx, item2idx, idx2item, train_user_items)
    ram_guard('ALS')
    return _LAZY['als']

def _load_bm25():
    if 'bm25' in _LAZY: return _LAZY['bm25']
    print('  [LAZY] Loading BM25 index...')
    with open(BM25_PATH, 'rb') as f: bm25_pack = pickle.load(f)
    bm25_obj = bm25_pack['bm25']
    bm25_ids = bm25_pack['appids'].astype(str)
    if 'tokenized_docs' in bm25_pack: del bm25_pack['tokenized_docs']
    del bm25_pack
    gc.collect()
    _LAZY['bm25'] = (bm25_obj, bm25_ids)
    ram_guard('BM25')
    return _LAZY['bm25']

def _load_sasrec():
    if 'sasrec' in _LAZY: return _LAZY['sasrec']
    if not os.path.exists(TEXT_SASREC_MODEL_PATH):
        _LAZY['sasrec'] = None
        return None
    print('  [LAZY] Loading SASRec model...')
    _, _, item2idx_s, _, _ = _load_als()
    raw_text_emb = np.load(GAME_TEXT_EMB_PATH, mmap_mode='r')
    num_items_sasrec = len(item2idx_s)
    emb_dim = raw_text_emb.shape[1]
    # Vectorized: build index mapping then use numpy advanced indexing
    appid_to_emb_idx = {str(aid): i for i, aid in enumerate(faiss_text_appids)}
    text_emb_matrix = np.zeros((num_items_sasrec + 1, emb_dim), dtype=np.float32)
    filled = 0
    for appid, idx_in_sasrec in item2idx_s.items():
        if appid in appid_to_emb_idx:
            text_emb_matrix[idx_in_sasrec + 1] = raw_text_emb[appid_to_emb_idx[appid]]
            filled += 1
    del raw_text_emb, appid_to_emb_idx
    gc.collect()

    class TextEnhancedSASRec(nn.Module):
        def __init__(self, num_items, max_len, hidden_dim, num_heads, num_layers, dropout, text_emb_matrix, freeze_text=True):
            super().__init__()
            self.num_items = num_items
            self.max_len = max_len
            self.hidden_dim = hidden_dim
            self.item_id_embedding = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
            self.position_embedding = nn.Embedding(max_len, hidden_dim)
            text_tensor = torch.tensor(text_emb_matrix, dtype=torch.float32)
            self.text_embedding = nn.Embedding.from_pretrained(text_tensor, freeze=freeze_text, padding_idx=0)
            self.text_projection = nn.Linear(text_tensor.shape[1], hidden_dim)
            encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, dim_feedforward=hidden_dim * 4, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.layer_norm = nn.LayerNorm(hidden_dim)
            self.dropout = nn.Dropout(dropout)
        def get_item_repr(self, item_ids):
            return self.item_id_embedding(item_ids) + self.text_projection(self.text_embedding(item_ids))
        def forward(self, input_ids):
            batch_size, seq_len = input_ids.shape
            device = input_ids.device
            pos = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
            x = self.get_item_repr(input_ids) + self.position_embedding(pos)
            x = self.layer_norm(self.dropout(x))
            causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)
            return self.encoder(x, mask=causal_mask, src_key_padding_mask=input_ids.eq(0))
        @torch.no_grad()
        def full_sort_scores(self, seq_tokens):
            self.eval()
            seq = seq_tokens[-self.max_len:]
            input_ids = torch.zeros((1, self.max_len), dtype=torch.long, device=next(self.parameters()).device)
            if len(seq) > 0:
                input_ids[0, -len(seq):] = torch.tensor(seq, dtype=torch.long, device=input_ids.device)
            h = self.forward(input_ids)
            nonzero = input_ids[0].ne(0).nonzero(as_tuple=False)
            last_h = h[0, -1] if len(nonzero) == 0 else h[0, nonzero[-1].item()]
            all_emb = self.get_item_repr(torch.arange(1, self.num_items + 1, device=input_ids.device))
            return torch.matmul(all_emb, last_h).detach().cpu().numpy()

    checkpoint = torch.load(TEXT_SASREC_MODEL_PATH, map_location=DEVICE)
    sasrec_config = checkpoint['config']
    model = TextEnhancedSASRec(
        num_items=sasrec_config['num_items'], max_len=sasrec_config['max_len'],
        hidden_dim=sasrec_config['hidden_dim'], num_heads=sasrec_config['num_heads'],
        num_layers=sasrec_config['num_layers'], dropout=sasrec_config['dropout'],
        text_emb_matrix=text_emb_matrix
    ).to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    del checkpoint, text_emb_matrix
    gc.collect()
    _LAZY['sasrec'] = model
    ram_guard('SASRec')
    return model

# ================================================================
# OPTIONAL MODELS (load at startup if small, else lazy)
# ================================================================
print('[OPT] Reranker V3...')
reranker_v3 = None
if os.path.exists(RERANKER_V3_MODEL_PATH):
    reranker_v3 = lgb.Booster(model_file=RERANKER_V3_MODEL_PATH)

# --- NSFW blacklist (lazy-loaded on first request) ---
NSFW_PATTERNS = ['nudity', 'sexual content', 'hentai', 'adult only', 'nsfw', 'sex']
GLOBAL_NSFW_APPIDS = None  # Will be loaded on first request

# Global trending cache
GLOBAL_TRENDING_DF = None
GLOBAL_TREND_VECS = None

gc.collect()
elapsed = time.time() - load_start
used_gb = get_ram_gb()
print(f'\n[OK] Essential models loaded in {elapsed:.1f}s')
print(f'     RAM used: {used_gb:.1f}GB / 15GB')
print(f'     FAISS game: {faiss_game_index.ntotal:,}')
print(f'     FAISS text: {faiss_text_index.ntotal if faiss_text_index else 0:,}')
print(f'     ALS/BM25/SASRec/NSFW: LAZY (will load on first request)')
print(f'     Reranker V3: {"loaded" if reranker_v3 else "N/A"}')

In [ ]:
# CELL 3: LIVE DATA COLLECTION & TEXT PREPROCESSING

def clean_game_text(game_name, genres, categories):
    raw_text = f"{game_name} {genres} {categories}"
    clean_text = raw_text.replace(',', ' ').replace('Free To Play', 'F2P').replace('Free to Play', 'F2P')
    for word in ['Steam Achievements', 'Steam Trading Cards', 'In-App Purchases', 'Steam Cloud',
        'Family Sharing', 'Captions available', 'Profile Features Limited', 'Includes level editor',
        'Remote Play on Phone', 'Remote Play on Tablet', 'Remote Play on TV', 'Remote Play Together']:
        clean_text = clean_text.replace(word, '')
    return " ".join(clean_text.split())

def get_entire_wishlist(steam_id):
    url = "https://api.steampowered.com/IWishlistService/GetWishlist/v1/"
    try:
        resp = requests.get(url, params={'steamid': steam_id, 'key': STEAM_API_KEY}, timeout=10).json()
        return [str(item['appid']) for item in resp.get('response', {}).get('items', [])]
    except: return []

def get_entire_library(steam_id):
    url = f"http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/?key={STEAM_API_KEY}&steamid={steam_id}&include_appinfo=1&include_played_free_games=1&format=json"
    library_dict = {}
    try:
        resp = requests.get(url, timeout=5).json()
        if 'response' in resp and 'games' in resp['response']:
            for g in resp['response']['games']:
                library_dict[str(g['appid'])] = g.get('playtime_forever', 0) + (g.get('playtime_2weeks', 0) * 5)
    except: pass
    return library_dict

def fetch_web_game_data(appid, game_name="Unknown", session=None):
    req = session if session else requests
    genres_text, tags_text = "", ""
    try:
        resp = req.get(f"https://steamspy.com/api.php?request=appdetails&appid={appid}", timeout=5).json()
        if resp:
            if 'genre' in resp and resp['genre']: genres_text = resp['genre']
            if 'name' in resp and resp['name']: game_name = resp['name']
            if 'tags' in resp and isinstance(resp['tags'], dict): tags_text = " ".join(list(resp['tags'].keys())[:20])
    except: pass
    if not genres_text or not tags_text:
        try:
            resp = req.get(f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=us&l=english", timeout=5).json()
            if resp and resp.get(str(appid), {}).get('success'):
                data = resp[str(appid)]['data']
                game_name = data.get('name', game_name)
                if not genres_text and 'genres' in data: genres_text = " ".join([c['description'] for c in data['genres']])
                if not tags_text and 'categories' in data: tags_text = " ".join([c['description'] for c in data['categories']])
        except: pass
    return {'appid': str(appid), 'name': game_name, 'genres': genres_text, 'text_data': clean_game_text(game_name, genres_text, tags_text)}

def get_trending_candidates_with_features(blacklist_appids, pool_size=500):
    print(f"-> [Trending] Fetching {pool_size} Popular New games...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36'}
    candidates = []
    offset = 0
    with requests.Session() as session:
        session.headers.update(headers)
        while offset < pool_size:
            url = f"https://store.steampowered.com/search/results/?filter=popularnew&sort_by=Released_DESC&os=win&start={offset}&count=50&infinite=1"
            try:
                res = session.get(url, timeout=10)
                if res.status_code != 200: break
                html = res.json().get('results_html', '')
                if not html: break
                soup = BeautifulSoup(html, 'html.parser')
                rows = soup.find_all('a', class_='search_result_row')
                if not rows: break
                for row in rows:
                    appid = row.get('data-ds-appid')
                    title_elem = row.find('span', class_='title')
                    if appid and title_elem:
                        if ',' in appid: appid = appid.split(',')[0]
                        if str(appid) not in blacklist_appids:
                            candidates.append({'appid': str(appid), 'name': title_elem.text.strip()})
                offset += 50
                time.sleep(0.5)
            except: break
    df_raw = pd.DataFrame(candidates).drop_duplicates(subset=['appid']).head(pool_size)
    if df_raw.empty:
        print('   [!] IP blocked. Using local DB fallback...')
        bl_str = ", ".join([f"'{x}'" for x in blacklist_appids]) if blacklist_appids else "''"
        try:
            db_fallback = con.execute(f"SELECT CAST(a.appid AS VARCHAR) as appid, a.name, a.genres, a.categories FROM read_parquet('{GAME_API_PATH}') a JOIN read_parquet('{GAME_FEAT_PATH}') e ON CAST(a.appid AS BIGINT) = e.appid WHERE CAST(a.appid AS VARCHAR) NOT IN ({bl_str}) ORDER BY e.total_reviews DESC NULLS LAST LIMIT {pool_size}").df()
            live_metadata = []
            for _, row in db_fallback.iterrows():
                txt = clean_game_text(row['name'] or "", row['genres'] or "", row['categories'] or "")
                if txt.strip(): live_metadata.append({'appid': str(row['appid']), 'name': row['name'], 'text_data': txt})
            del db_fallback
            return pd.DataFrame(live_metadata)
        except: return pd.DataFrame()
    live_metadata = []
    try:
        appids_str = ", ".join([f"'{x}'" for x in df_raw['appid']])
        db_df = con.execute(f"SELECT CAST(appid AS VARCHAR) as appid, name, genres, categories FROM read_parquet('{GAME_API_PATH}') WHERE CAST(appid AS VARCHAR) IN ({appids_str})").df()
        db_found = set()
        for _, row in db_df.iterrows():
            aid = str(row['appid'])
            db_found.add(aid)
            text_data = clean_game_text(row['name'] or "", row['genres'] or "", row['categories'] or "")
            if text_data.strip(): live_metadata.append({'appid': aid, 'name': row['name'], 'genres': row['genres'], 'text_data': text_data})
        del db_df
        appids_to_fetch_web = [row for _, row in df_raw.iterrows() if str(row['appid']) not in db_found]
    except:
        appids_to_fetch_web = [row for _, row in df_raw.iterrows()]
    if appids_to_fetch_web:
        with requests.Session() as thread_session:
            with ThreadPoolExecutor(max_workers=10) as executor:
                futures = {executor.submit(fetch_web_game_data, row['appid'], row['name'], thread_session): row['appid'] for row in appids_to_fetch_web}
                for future in as_completed(futures):
                    try:
                        res = future.result()
                        if res['text_data'].strip(): live_metadata.append(res)
                    except: pass
    del df_raw
    gc.collect()
    return pd.DataFrame(live_metadata)

print('[OK] Live data collection ready.')

In [ ]:
# CELL 4: CORE HYBRID RECOMMENDATION ENGINE
# Optimized: no local_con, lazy NSFW, np.dot for cosine, DuckDB-only user queries

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\u00C0-\u024F\s]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().split()

def build_game_token_sets_if_needed():
    global GAME_TOKEN_SETS, GAME_TOKEN_SETS_BUILT
    if GAME_TOKEN_SETS_BUILT:
        return
    print('Building GAME_TOKEN_SETS (lazy)...')
    gd = pd.read_parquet(GAME_DOCS_PATH, columns=['appid', 'genres', 'categories'])
    gd['appid'] = gd['appid'].astype(str)
    # Vectorized token extraction
    texts = (gd['genres'].fillna('') + ' ' + gd['categories'].fillna('')).str.lower()
    token_lists = texts.str.findall(r'\w+')
    GAME_TOKEN_SETS = dict(zip(gd['appid'], [set(t) for t in token_lists]))
    del gd, texts, token_lists
    gc.collect()
    GAME_TOKEN_SETS_BUILT = True
    print(f'  Built for {len(GAME_TOKEN_SETS):,} games')

def get_history_token_set(author_id):
    key = str(author_id)
    if key in HISTORY_TOKEN_CACHE:
        return HISTORY_TOKEN_CACHE[key]
    author_id_sql = f"'{str(author_id).replace(chr(39), chr(39)+chr(39))}'"
    try:
        df = con.execute(f"SELECT CAST(appid AS VARCHAR) AS appid FROM read_parquet('{TRAIN_PATH}') WHERE CAST(author_id AS VARCHAR) = {author_id_sql} ORDER BY reverse_chron_rank ASC LIMIT 50").df()
        hist = df['appid'].astype(str).tolist()
        del df
    except:
        hist = []
    build_game_token_sets_if_needed()
    s = set()
    for aid in hist:
        s.update(GAME_TOKEN_SETS.get(aid, set()))
    HISTORY_TOKEN_CACHE[key] = s
    return s

def is_nsfw_game(appid):
    text = appid_to_tag_text.get(str(appid), '').lower()
    name = appid_to_name.get(str(appid), '').lower()
    return any(p in (text + ' ' + name) for p in NSFW_PATTERNS)

def _load_nsfw_blacklist():
    """Lazy-load NSFW blacklist from DuckDB (avoids startup query)"""
    global GLOBAL_NSFW_APPIDS
    if GLOBAL_NSFW_APPIDS is not None:
        return GLOBAL_NSFW_APPIDS
    print('  [LAZY] Loading NSFW blacklist...')
    try:
        nsfw_df = con.execute(
            f"SELECT CAST(appid AS VARCHAR) as appid FROM read_parquet('{GAME_API_PATH}') "
            "WHERE genres ILIKE '%Nudity%' OR genres ILIKE '%Sexual Content%' "
            "OR genres ILIKE '%Hentai%' OR genres ILIKE '%NSFW%' OR genres ILIKE '%Adult Only%' "
            "OR categories ILIKE '%Nudity%' OR categories ILIKE '%Sexual Content%' "
            "OR name ILIKE '%Hentai%' OR name ILIKE '%Sex%'"
        ).df()
        GLOBAL_NSFW_APPIDS = set(nsfw_df['appid'].tolist())
        del nsfw_df
    except:
        GLOBAL_NSFW_APPIDS = set()
    return GLOBAL_NSFW_APPIDS

def build_rerank_features(cand_df, author_id=None):
    if cand_df is None or cand_df.empty:
        return pd.DataFrame(), []
    df = cand_df.copy()
    df['appid'] = df['appid'].astype(str)
    df['author_id'] = str(author_id) if author_id else ''
    source_cols = ['dense_score', 'bm25_score', 'als_score', 'i2i_score', 'sasrec_score', 'popular_score', 'final_blend']
    for c in source_cols:
        if c not in df.columns: df[c] = 0.0
    # Game features already from DuckDB query — do NOT overwrite with GAME_NUM_DICT
    # Query DuckDB for user features (single user, not full table)
    USER_NUM_COLS = ['total_reviews_written', 'max_games_owned', 'total_lifetime_reviews', 'avg_playtime_all_games', 'avg_playtime_before_review', 'steam_purchase_ratio', 'deck_usage_ratio']
    if author_id:
        try:
            udf = con.execute(f"SELECT CAST(author_id AS VARCHAR) as author_id, {', '.join(USER_NUM_COLS)} FROM read_parquet('{USER_FEAT_PATH}') WHERE CAST(author_id AS VARCHAR) = '{author_id}'").df()
            if not udf.empty:
                for c in USER_NUM_COLS:
                    if c not in udf.columns: udf[c] = 0.0
                    df[c] = pd.to_numeric(udf[c].iloc[0], errors='coerce').fillna(0.0)
                del udf
            else:
                for c in USER_NUM_COLS: df[c] = 0.0
        except:
            for c in USER_NUM_COLS: df[c] = 0.0
    else:
        for c in USER_NUM_COLS: df[c] = 0.0
    for c in ['total_reviews', 'positive_review_ratio', 'avg_weighted_score', 'global_avg_playtime', 'avg_review_length', 'deep_review_ratio'] + USER_NUM_COLS:
        if c not in df.columns: df[c] = 0.0
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)
    df['playtime_diff_abs'] = np.abs(df['avg_playtime_all_games'].fillna(0.0) - df['global_avg_playtime'].fillna(0.0))
    build_game_token_sets_if_needed()
    overlaps, jaccards = [], []
    for row in df[['author_id', 'appid']].itertuples(index=False):
        hset = get_history_token_set(row.author_id) if row.author_id else set()
        gset = GAME_TOKEN_SETS.get(str(row.appid), set())
        if not hset or not gset:
            overlaps.append(0); jaccards.append(0.0)
        else:
            inter = len(hset & gset)
            overlaps.append(inter); jaccards.append(inter / max(len(hset | gset), 1))
    df['history_token_overlap'] = overlaps
    df['history_token_jaccard'] = jaccards
    df['sasrec_x_dense'] = df['sasrec_score'] * df['dense_score']
    df['sasrec_x_als'] = df['sasrec_score'] * df['als_score']
    df['dense_x_bm25'] = df['dense_score'] * df['bm25_score']
    feature_cols = source_cols + ['total_reviews', 'positive_review_ratio', 'avg_weighted_score', 'global_avg_playtime', 'avg_review_length', 'deep_review_ratio',
                                  'total_reviews_written', 'max_games_owned', 'total_lifetime_reviews', 'avg_playtime_all_games', 'avg_playtime_before_review',
                                  'steam_purchase_ratio', 'deck_usage_ratio', 'playtime_diff_abs', 'history_token_overlap', 'history_token_jaccard',
                                  'sasrec_x_dense', 'sasrec_x_als', 'dense_x_bm25']
    for c in feature_cols:
        if c not in df.columns: df[c] = 0.0
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)
    return df, feature_cols


def get_perfect_hybrid_recommendation(author_id, custom_query="", top_k=10, new_games_k=10, session_exclude=None, filter_nsfw=True):
    start_time = time.time()
    if session_exclude is None: session_exclude = []
    author_id_str = str(author_id)

    # NSFW blacklist (lazy-loaded once)
    nsfw_set = _load_nsfw_blacklist() if filter_nsfw else set()

    # Steam API
    wishlist_appids = get_entire_wishlist(author_id_str)
    library_data = get_entire_library(author_id_str)
    blacklist_appids = set(wishlist_appids + list(library_data.keys()) + session_exclude)
    if filter_nsfw: blacklist_appids.update(nsfw_set)
    is_private = not wishlist_appids and not library_data

    # User taste vector
    user_vector = None
    if not is_private:
        all_ids = list(set(list(library_data.keys()) + wishlist_appids))
        all_ids_str = ", ".join([f"'{x}'" for x in all_ids]) if all_ids else "''"
        try:
            raw_text_df = con.execute(f"SELECT CAST(appid AS VARCHAR) as appid, name, genres, categories FROM read_parquet('{GAME_API_PATH}') WHERE CAST(appid AS VARCHAR) IN ({all_ids_str})").df()
            text_dict = {str(row['appid']): clean_game_text(row['name'] or "", row['genres'] or "", row['categories'] or "") for _, row in raw_text_df.iterrows()}
            del raw_text_df
            text_weight_map = {}
            for aid in list(library_data.keys()):
                if aid in text_dict:
                    pt = library_data.get(aid, 0) / 60
                    w = 1 if pt <= 0 else min(int(np.log1p(pt) * 2) + 2, 9)
                    if text_dict[aid].strip(): text_weight_map[text_dict[aid]] = text_weight_map.get(text_dict[aid], 0) + w
            for aid in wishlist_appids:
                if aid in text_dict and text_dict[aid].strip():
                    text_weight_map[text_dict[aid]] = text_weight_map.get(text_dict[aid], 0) + 20
            del text_dict
            if text_weight_map:
                u_embs = nlp_model.encode(list(text_weight_map.keys()), normalize_embeddings=True, batch_size=32)
                u_weights = np.array(list(text_weight_map.values())).reshape(-1, 1)
                user_vector = (np.sum(u_embs * u_weights, axis=0) / np.sum(u_weights)).reshape(1, -1)
                del u_embs, u_weights
        except: pass

    has_query = bool(custom_query and custom_query.strip())
    query_vec = nlp_model.encode([custom_query.strip()], normalize_embeddings=True) if has_query else None

    # TRENDING
    global GLOBAL_TRENDING_DF, GLOBAL_TREND_VECS
    if GLOBAL_TRENDING_DF is None or len(GLOBAL_TRENDING_DF) == 0:
        GLOBAL_TRENDING_DF = get_trending_candidates_with_features(blacklist_appids=set(), pool_size=1000)
        if not GLOBAL_TRENDING_DF.empty:
            GLOBAL_TREND_VECS = nlp_model.encode(GLOBAL_TRENDING_DF['text_data'].tolist(), normalize_embeddings=True, batch_size=32)

    trending_df = pd.DataFrame()
    if GLOBAL_TRENDING_DF is not None and not GLOBAL_TRENDING_DF.empty:
        valid_mask = ~GLOBAL_TRENDING_DF['appid'].isin(blacklist_appids)
        trending_df = GLOBAL_TRENDING_DF.loc[valid_mask]  # view, not copy
        if not trending_df.empty:
            t_vecs = GLOBAL_TREND_VECS[valid_mask.values]
            trending_df = trending_df.copy()  # copy only when we modify
            trending_df['steam_link'] = "https://store.steampowered.com/app/" + trending_df['appid'].astype(str)
            if has_query:
                sims = np.clip(np.dot(query_vec, t_vecs.T)[0], 0, 1)  # np.dot faster than cosine_similarity
                if user_vector is not None:
                    usims = np.clip(np.dot(user_vector, t_vecs.T)[0], 0, 1)
                    sims = sims * (1.0 + usims * 0.30)
            elif user_vector is not None:
                sims = np.dot(user_vector, t_vecs.T)[0]
            else:
                sims = np.full(len(trending_df), np.nan)
            if not np.all(np.isnan(sims)):
                mn, mx = np.nanmin(sims), np.nanmax(sims)
                trending_df['match_probability'] = ((sims - mn) / (mx - mn + 1e-9) * 0.40) + 0.55 if mx > mn else 0.80
            else:
                trending_df['match_probability'] = np.nan
            trending_df = trending_df.sort_values('match_probability', ascending=False).head(new_games_k)
            del t_vecs

    # CATALOG
    catalog_df = pd.DataFrame()
    if is_private and not has_query:
        catalog_df = con.execute(f"SELECT CAST(appid AS VARCHAR) as appid, NULL as match_probability FROM read_parquet('{GAME_FEAT_PATH}') WHERE total_reviews > 10000 ORDER BY avg_weighted_score DESC LIMIT {top_k}").df()
    else:
        try:
            user_info = con.execute(f"SELECT COALESCE(CAST(primary_language AS VARCHAR), 'english') as primary_language, CAST(COALESCE(avg_playtime_all_games, 10.0) AS REAL) as u_avg_playtime, CAST(COALESCE(deck_usage_ratio, 0.0) AS REAL) as deck_usage, CAST(COALESCE(total_reviews_written, 1.0) AS REAL) as total_reviews FROM read_parquet('{USER_FEAT_PATH}') WHERE CAST(author_id AS VARCHAR) = '{author_id_str}'").df()
            u_lang, u_playtime, u_deck, u_rev = ('english', 10.0, 0.0, 1.0) if len(user_info) == 0 else (user_info['primary_language'].iloc[0], user_info['u_avg_playtime'].iloc[0], user_info['deck_usage'].iloc[0], user_info['total_reviews'].iloc[0])
            del user_info

            # Fetch user history FIRST (needed for Two-Tower + SASRec + I2I)
            hist_appids = []
            try:
                hist_df = con.execute(f"SELECT CAST(appid AS VARCHAR) AS appid FROM read_parquet('{TRAIN_PATH}') WHERE CAST(author_id AS VARCHAR) = '{author_id_str}' ORDER BY reverse_chron_rank ASC LIMIT 30").df()
                hist_appids = hist_df['appid'].astype(str).tolist()
                del hist_df
            except: pass

            # Two-Tower — model expects history_appids
            hist_ids_for_tt = hist_appids[:50] if hist_appids else ['']
            user_emb = user_model({
                "primary_language": tf.constant([u_lang]),
                "u_avg_playtime": tf.constant([u_playtime], dtype=tf.float32),
                "deck_usage": tf.constant([u_deck], dtype=tf.float32),
                "total_reviews": tf.constant([u_rev], dtype=tf.float32),
                "appid": tf.constant([" "]),
                "categories": tf.constant([" "]),
                "pos_review_ratio": tf.constant([0.0], dtype=tf.float32),
                "g_avg_playtime": tf.constant([0.0], dtype=tf.float32),
                "history_appids": tf.constant(hist_ids_for_tt)
            })
            distances_tt, indices_tt = faiss_game_index.search(user_emb.numpy(), 2000)
            faiss_dict = {str(faiss_game_appids[x]): float(distances_tt[0][i]) for i, x in enumerate(indices_tt[0])}
            del user_emb, distances_tt, indices_tt

            # User taste
            user_taste_dict = {}
            if user_vector is not None and faiss_text_index is not None:
                dist_u, ind_u = faiss_text_index.search(user_vector, min(5000, faiss_text_index.ntotal))
                user_taste_dict = {str(faiss_text_appids[x]): float(dist_u[0][i]) for i, x in enumerate(ind_u[0])}
                del dist_u, ind_u

            # RecSas branches
            als_dict, bm25_dict, i2i_dict, sasrec_dict = {}, {}, {}, {}

            # ALS (lazy-loaded on first use)
            als_model_l, user2idx_l, item2idx_l, idx2item_l, train_user_items_l = _load_als()
            if author_id_str in user2idx_l:
                try:
                    recs = als_model_l.recommend(userid=user2idx_l[author_id_str], user_items=train_user_items_l[user2idx_l[author_id_str]], N=1500, filter_already_liked_items=True)
                    if isinstance(recs, tuple): item_indices, als_scores = recs
                    else: item_indices, als_scores = zip(*recs)
                    als_dict = {str(idx2item_l[int(i)]): float(s) for i, s in zip(item_indices, als_scores) if 0 <= int(i) < len(idx2item_l)}
                    del recs
                except: pass

            # BM25 (lazy-loaded on first use)
            if has_query:
                try:
                    bm25_l, bm25_appids_l = _load_bm25()
                    toks = simple_tokenize(custom_query)
                    if toks:
                        bm25_scores = bm25_l.get_scores(toks)
                        top_idx = np.argpartition(-bm25_scores, 700)[:700]
                        bm25_dict = {str(bm25_appids_l[i]): float(bm25_scores[i]) for i in top_idx[np.argsort(-bm25_scores[top_idx])][:700]}
                        del bm25_scores, top_idx
                except: pass

            # I2I + SASRec (use hist_appids fetched above)
            if hist_appids:
                try:
                    hist_str = ", ".join([f"'{x}'" for x in hist_appids])
                    i2i_df = con.execute(f"SELECT CAST(target_appid AS VARCHAR) AS appid, SUM(i2i_score) AS i2i_score FROM read_parquet('{ITEM2ITEM_PATH}') WHERE CAST(source_appid AS VARCHAR) IN ({hist_str}) GROUP BY target_appid ORDER BY i2i_score DESC LIMIT 700").df()
                    i2i_dict = dict(zip(i2i_df['appid'].astype(str), i2i_df['i2i_score'].astype(float)))
                    del i2i_df
                except: pass

            # SASRec (lazy-loaded on first use)
            sasrec_model_l = _load_sasrec()
            if sasrec_model_l is not None and hist_appids:
                try:
                    _, _, item2idx_s, idx2item_s, _ = _load_als()  # cached, no extra RAM
                    seq_tokens = [item2idx_s[str(a)] + 1 for a in reversed(hist_appids) if str(a) in item2idx_s]
                    if seq_tokens:
                        scores = sasrec_model_l.full_sort_scores(seq_tokens)
                        n = min(700, len(scores))
                        top_idx = np.argpartition(-scores, n)[:n]
                        sasrec_dict = {str(idx2item_s[i]): float(scores[i]) for i in top_idx[np.argsort(-scores[top_idx])][:n]}
                        del scores, top_idx
                except: pass

            # Text search (FAISS text — always loaded)
            semantic_dict = {}
            if has_query and faiss_text_index is not None:
                dist_q, ind_q = faiss_text_index.search(query_vec, 3000)
                semantic_dict = {str(faiss_text_appids[x]): float(dist_q[0][i]) for i, x in enumerate(ind_q[0])}
                del dist_q, ind_q

            # Combine ALL candidates
            all_ids = set()
            for d in [faiss_dict, semantic_dict, user_taste_dict, als_dict, bm25_dict, i2i_dict, sasrec_dict]:
                for aid in d:
                    if aid not in blacklist_appids: all_ids.add(aid)

            filtered = list(all_ids)[:max(300, top_k * 10)]
            del all_ids

            if filtered:
                f_str = ", ".join([f"'{x}'" for x in filtered])
                df_f = con.execute(f"SELECT CAST(c.appid AS VARCHAR) as appid, CAST(COALESCE(g.positive_review_ratio, 0.0) AS REAL) as positive_review_ratio, CAST(COALESCE(g.avg_weighted_score, 0.0) AS REAL) as avg_weighted_score, CAST(COALESCE(g.total_reviews, 0) AS REAL) as total_reviews, CAST(COALESCE(u.avg_playtime_all_games, {u_playtime}) AS REAL) as avg_playtime_all_games, CAST(COALESCE(u.steam_purchase_ratio, 0.5) AS REAL) as steam_purchase_ratio, CAST(COALESCE(u.total_reviews_written, {u_rev}) AS REAL) as total_reviews_written, CAST(COALESCE(u.max_games_owned, 10) AS REAL) as max_games_owned, CAST(COALESCE(g.avg_review_length, 0.0) AS REAL) as avg_review_length, CAST(COALESCE(g.deep_review_ratio, 0.0) AS REAL) as deep_review_ratio, CAST(ABS(COALESCE(u.avg_playtime_all_games, {u_playtime}) - COALESCE(g.global_avg_playtime, 0)) / (COALESCE(g.global_avg_playtime, 1) + 1) AS REAL) as playtime_diff_ratio, CAST(COALESCE(g.positive_review_ratio, 0) * COALESCE(g.avg_weighted_score, 0) AS REAL) as quality_score FROM (SELECT unnest([{f_str}]) as appid) as c LEFT JOIN read_parquet('{GAME_FEAT_PATH}') g ON CAST(c.appid AS BIGINT) = g.appid LEFT JOIN read_parquet('{USER_FEAT_PATH}') u ON CAST(u.author_id AS VARCHAR) = '{author_id_str}'").df()
                del filtered, f_str

                df_f['raw_lgbm'] = ranker.predict(df_f[['avg_playtime_all_games', 'steam_purchase_ratio', 'total_reviews_written', 'max_games_owned', 'positive_review_ratio', 'avg_weighted_score', 'avg_review_length', 'deep_review_ratio', 'total_reviews', 'playtime_diff_ratio', 'quality_score']])
                df_f['faiss_score'] = df_f['appid'].map(faiss_dict).fillna(0)
                df_f['user_taste_score'] = df_f['appid'].map(user_taste_dict).fillna(0)
                df_f['als_score'] = df_f['appid'].map(als_dict).fillna(0)
                df_f['bm25_score'] = df_f['appid'].map(bm25_dict).fillna(0)
                df_f['i2i_score'] = df_f['appid'].map(i2i_dict).fillna(0)
                df_f['sasrec_score'] = df_f['appid'].map(sasrec_dict).fillna(0)
                df_f['dense_score'] = df_f['appid'].map(semantic_dict).fillna(0)
                df_f['popular_score'] = 0.0
                del faiss_dict, user_taste_dict, als_dict, bm25_dict, i2i_dict, sasrec_dict, semantic_dict

                # Normalize
                for c in ['faiss_score', 'user_taste_score', 'als_score', 'bm25_score', 'i2i_score', 'sasrec_score', 'dense_score', 'raw_lgbm']:
                    mn, mx = df_f[c].min(), df_f[c].max()
                    df_f[f'n_{c}'] = (df_f[c] - mn) / (mx - mn + 1e-9) if mx > mn else 0.0

                # Hybrid scoring
                if has_query:
                    df_f['hybrid'] = df_f['n_dense_score'] * (1.0 + df_f['n_user_taste_score'] * 0.30 + df_f['n_raw_lgbm'] * 0.10)
                else:
                    df_f['hybrid'] = df_f['n_user_taste_score'] * (1.0 + df_f['n_faiss_score'] * 0.30 + df_f['n_raw_lgbm'] * 0.15)
                df_f['hybrid'] += 0.10 * df_f['n_als_score'] + 0.05 * df_f['n_bm25_score'] + 0.08 * df_f['n_i2i_score'] + 0.12 * df_f['n_sasrec_score']

                # Reranker V3
                if reranker_v3 is not None and author_id_str:
                    feat_df, feature_cols = build_rerank_features(df_f, author_id=author_id_str)
                    if not feat_df.empty and len(feature_cols) > 0:
                        for c in feature_cols:
                            if c not in feat_df.columns: feat_df[c] = 0.0
                        df_f['rerank_score'] = reranker_v3.predict(feat_df[feature_cols])
                        rn_min, rn_max = df_f['rerank_score'].min(), df_f['rerank_score'].max()
                        df_f['n_rerank'] = (df_f['rerank_score'] - rn_min) / (rn_max - rn_min + 1e-9) if rn_max > rn_min else 0.5
                        df_f['hybrid'] = 0.6 * df_f['n_rerank'] + 0.4 * df_f['hybrid']
                        del feat_df

                df_f['match_probability'] = (df_f['hybrid'] - df_f['hybrid'].min()) / (df_f['hybrid'].max() - df_f['hybrid'].min() + 1e-9) * 0.20 + 0.75
                catalog_df = df_f.sort_values('match_probability', ascending=False).head(top_k)
                del df_f
        except Exception as e:
            traceback.print_exc()

    if not catalog_df.empty:
        c_ids = ", ".join([f"'{x}'" for x in catalog_df['appid']])
        meta = con.execute(f"SELECT CAST(appid AS VARCHAR) as appid, name, genres, categories FROM read_parquet('{GAME_API_PATH}') WHERE CAST(appid AS VARCHAR) IN ({c_ids})").df()
        meta['tags_display'] = meta['genres'].fillna('') + " , " + meta['categories'].fillna('')
        catalog_df = pd.merge(catalog_df[['appid', 'match_probability']], meta, on='appid', how='inner')
        catalog_df['steam_link'] = "https://store.steampowered.com/app/" + catalog_df['appid'].astype(str)
        del meta

    gc.collect()
    print(f"Done! {time.time() - start_time:.2f}s")

    return {
        'catalog': catalog_df[['name', 'match_probability', 'tags_display', 'steam_link']] if not catalog_df.empty else pd.DataFrame(),
        'trending': trending_df[['name', 'match_probability', 'text_data', 'steam_link']] if not trending_df.empty else pd.DataFrame()
    }

print('[OK] Core engine ready.')

In [ ]:
# CELL 5: FASTAPI SERVER

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List
import nest_asyncio
import uvicorn

nest_asyncio.apply()

class GameResult(BaseModel):
    name: str
    steam_link: str
    match_probability: float

class APIResponse(BaseModel):
    status: str
    message: str = ""
    results: List[GameResult] = []

app = FastAPI(title="Steam Hybrid AI API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def read_root():
    return {"message": "Server AI Recommendation dang hoat dong!"}

@app.get("/api/recommend", response_model=APIResponse)
def get_api_recommendations(user_id: str = "", query: str = "", top_k: int = 10, exclude: str = "", nsfw: bool = True):
    if not user_id and not query:
        return {"status": "error", "message": "Vui long nhap Steam ID hoac The loai ban muon!", "results": []}
    try:
        exclude_list = [x.strip() for x in exclude.split(",") if x.strip()]
        print(f"\n[REQUEST] ID: {user_id} | top_k: {top_k} | exclude: {len(exclude_list)} | nsfw: {nsfw}")
        half_k = max(1, top_k // 2)
        ai_results = get_perfect_hybrid_recommendation(
            author_id=user_id, custom_query=query, top_k=half_k + (top_k % 2), new_games_k=half_k, session_exclude=exclude_list, filter_nsfw=nsfw
        )
        final_list = []
        for df_key in ['catalog', 'trending']:
            df = ai_results.get(df_key)
            if df is not None and not df.empty:
                for _, row in df.iterrows():
                    prob = row.get('match_probability', 0)
                    final_list.append({
                        'name': str(row.get('name', 'Unknown')),
                        'steam_link': str(row.get('steam_link', '')),
                        'match_probability': 0.0 if pd.isna(prob) or np.isinf(prob) else float(prob)
                    })
        final_list.sort(key=lambda x: x['match_probability'], reverse=True)
        return {"status": "success", "message": "Phan tich thanh cong!", "results": final_list}
    except Exception as e:
        print(f"\n[ERROR]: {traceback.format_exc()}")
        return {"status": "error", "message": "Loi he thong", "results": []}

print('[OK] FastAPI ready.')

In [ ]:
# CELL 6: START SERVER + LOCALTUNNEL

import subprocess
import urllib.request

subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(1)

try:
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=5).read().decode().strip()
    print(f"='*60")
    print(f"IP: {ip}")
    print(f"='*60")
except: pass

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("\nDang tao localtunnel...")
!npx localtunnel --port 8000